# Reaktive Agenten mit OPC UA-Integration (Mesa 3.0.x)

Dieses Notebook verbindet z

-  **agentenbasierte Simulation** mit Mesa 3.0.x,
-  **industrielle Kommunikation** über OPC UA (asyncua).


## Übergeordnete Architektur

```
┌─────────────────────────────────────────────────────────┐
│                   Jupyter-Prozess                        │
│                                                          │
│  Mesa-Simulation (synchron)                              │
│  ┌─────────────────────────────┐                         │
│  │ MachineAgent.step()         │                         │
│  │   → schreibt in            │                         │
│  │     opcua_write_buffer      │──────┐                  │
│  └─────────────────────────────┘      │                  │
│                                       ▼                  │
│  asyncio Event-Loop (parallel)  ┌─────────────────────┐  │
│  ┌──────────────────────┐       │ opcua_write_buffer   │  │
│  │ opcua_writer()       │◄──────│ {"M01": {Temp, State,│  │
│  │   → write_value()    │       │          Busy}, ...} │  │
│  └──────────────────────┘       └─────────────────────┘  │
│  ┌──────────────────────┐                                 │
│  │ subscription_hot_    │                                 │
│  │ monitor()            │                                 │
│  │   → liest vom Server │                                 │
│  │   → setzt RepairNeeded│                                │
│  └──────────────────────┘                                 │
│                    │                                      │
└────────────────────┼──────────────────────────────────────┘
                     │ OPC UA (TCP)
                     ▼
        ┌─────────────────────────┐
        │  OPC UA Factory-Server  │
        │  M01/Temperature        │
        │  M01/State              │
        │  M01/Busy               │
        │  M01_RepairNeeded       │
        │  ...                    │
        └─────────────────────────┘
```


## 1. Imports und Ausgabe-Widget

Zuerst richten wir das `Output`-Widget ein und leiten `print` global um,
damit OPC UA-Ausgaben aus asyncio-Callbacks sichtbar werden.


In [1]:
# Importpfad: die .py-Module liegen in ../src (Jupyter startet in notebooks/)
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
if _src not in sys.path:
    sys.path.insert(0, _src)
print("src-Pfad:", _src)

src-Pfad: C:\Users\feris\Desktop\stuff\Julia_Agent\src


In [2]:
import asyncio
import builtins
import ipywidgets as widgets
from IPython.display import display

# Der OPC UA HOT-Monitor-Client muss als OPC_Hot_Monitor_Client_de.py
# im selben Verzeichnis liegen.
import OPC_Hot_Monitor_Client_de as hot_client

# ── Output-Widget für OPC UA Meldungen ────────────────────────────────────────
# Alle print()-Aufrufe – auch aus asyncio-Callbacks – landen hier.
opcua_out = widgets.Output(layout=widgets.Layout(
    height="150px",
    overflow_y="auto",
    border="1px solid gray",
))

_original_print = builtins.print
def _patched_print(*args, **kwargs):
    with opcua_out:
        _original_print(*args, **kwargs)
builtins.print = _patched_print

print("Mesa + OPC UA Umgebungs-Imports bereit.")


Event-Loop eingerichtet. Server-Endpunkt: opc.tcp://localhost:4840/freeopcua/server/
HOT-Schwellenwert: 60.0 °C


## 2. Gemeinsamer Datenpuffer

Der `opcua_write_buffer` ist das zentrale Bindeglied zwischen Mesa und OPC UA.

- **Mesa schreibt** nach jedem `step()` die aktuellen Agentenwerte hinein.
- **Der asyncio Writer-Task liest** den Puffer periodisch und überträgt die Werte
  an den OPC UA-Server.

Da Mesa und der Writer-Task im selben Python-Prozess laufen und der asyncio
Event-Loop kooperativ ist (kein echtes Multithreading), sind keine Locks
erforderlich – es gibt keine Race Conditions.

```
MachineAgent.step()          asyncio Writer-Task
      │                             │
      │  opcua_write_buffer         │
      │  ┌─────────────────────┐    │
      └─►│ "M01": {42.3, "WARM"│──► │ write_value() → OPC UA Server
         │ "M02": {71.1, "HOT" │    │
         └─────────────────────┘    │
```


In [3]:
# Gemeinsame Datenpuffer (Entkopplung Mesa <-> OPC UA) sind zentral in
# factory_model_de definiert. Wir importieren genau diese Objekte, damit
# Agenten (im Modul) und OPC-Tasks (hier im Notebook) denselben Zustand teilen.
from factory_model_de import (
    opcua_write_buffer,
    opcua_repair_buffer,
    opcua_earlywarning_buffer,
    server_flags,
)

## 3. Mesa-Agenten mit OPC UA-Anbindung

Gegenüber der ursprünglichen Version gibt es zwei Erweiterungen:

### 3.1 MachineAgent: `machine_name` und Puffer-Update

Jeder `MachineAgent` bekommt einen festen **OPC UA-Maschinennamen** (z.B. `"M01"`),
der dem Knoten auf dem Server entspricht. Am Ende jedes `step()` schreibt er
seine aktuellen Werte in den `opcua_write_buffer`.

**Warum am Ende von `step()` und nicht sofort auf den Server?**
OPC UA-Schreiboperationen sind `async` – sie können nicht direkt aus synchronem
Mesa-Code aufgerufen werden. Der Puffer ist die saubere Entkopplung.

### 3.2 FactoryModelExtended: Maschinen nummerieren

Damit jeder Agent einen eindeutigen Servernamen hat, werden die Maschinen beim
Erstellen durchnummeriert: `M01`, `M02`, ...


In [4]:
# Agenten sind nach src/factory_model_de.py ausgelagert (headless testbar mit pytest).
from factory_model_de import MachineAgent, MaintenanceAgent, OrderAgent

In [5]:
# Modell + Reporter-Funktionen aus dem ausgelagerten Modul importieren.
from factory_model_de import (
    FactoryModelExtended, machine_is_warned,
    count_hot_machines, count_warm_machines, count_machines,
    count_early_warnings, count_orders, count_maintenance,
    avg_temperature, max_temperature, prevention_rate,
)


## 4. Visualisierung


In [6]:
from mesa.visualization import SolaraViz, make_space_component
from mesa.visualization.components import AgentPortrayalStyle
from mesa.visualization.utils import update_counter
import solara
from matplotlib.figure import Figure
from matplotlib.lines import Line2D
import matplotlib.patheffects as pe

DENSITY       = 0.2
THRESHOLD     = 70
N_MAINTENANCE = 2    # fester Start mit 2 Wartungsagenten (live aenderbar)
N_ORDERS      = 10

# Farbzuordnung mit gutem Kontrast
ZUSTAND_FARBEN = {
    "HOT":  "#d62728",  # rot
    "WARM": "#ff7f0e",  # orange
    "COOL": "#1f77b4",  # blau
    "OK":   "#2ca02c",  # gruen
}
EARLYWARNING_FARBE = "gold"
WARTUNG_FARBE      = "#17becf"
AUFTRAG_FARBE      = "black"
# Textumrandung -> Beschriftung auf jeder Markerfarbe lesbar (entzerrt das Raster)
_STROKE = [pe.withStroke(linewidth=2.2, foreground="white")]


def agent_portrayal_ext(agent):
    # AgentPortrayalStyle statt dict -> keine FutureWarning (Mesa 3.5+)
    if isinstance(agent, MachineAgent):
        color = EARLYWARNING_FARBE if machine_is_warned(agent) else ZUSTAND_FARBEN.get(agent.state, "#2ca02c")
        return AgentPortrayalStyle(color=color, size=1100, marker="s",
                                   alpha=1.0, edgecolors="black", linewidths=1.0, zorder=1)
    if isinstance(agent, MaintenanceAgent):
        return AgentPortrayalStyle(color=WARTUNG_FARBE, size=520, marker="P",
                                   alpha=1.0, edgecolors="black", linewidths=1.2, zorder=3)
    if isinstance(agent, OrderAgent):
        return AgentPortrayalStyle(color=AUFTRAG_FARBE, size=150, marker="D",
                                   alpha=1.0, edgecolors="white", linewidths=0.8, zorder=2)
    return AgentPortrayalStyle()


def make_post_process_ext(model):
    def post_process(ax):
        ax.figure.set_size_inches(7, 7)
        legend = ax.get_legend()
        if legend:
            legend.remove()
        for txt in ax.texts[:]:
            txt.remove()
        for agent in model.agents:
            x, y = agent.pos
            if isinstance(agent, MachineAgent):
                label = (f"{agent.machine_name}\n{agent.temperature:.0f}°"
                         if agent.machine_name else f"{agent.temperature:.0f}°")
                ax.text(x, y, label, ha="center", va="center", fontsize=6.5,
                        color="black", fontweight="bold", zorder=5, path_effects=_STROKE)
            elif isinstance(agent, MaintenanceAgent):
                ax.text(x, y, "W", ha="center", va="center", fontsize=9,
                        color="black", fontweight="bold", zorder=5, path_effects=_STROKE)
        # Legende ausserhalb des Plots (Mesa zeichnet selbst keine)
        handles = [
            Line2D([0], [0], marker="s", color="w", markerfacecolor=ZUSTAND_FARBEN["OK"],
                   markeredgecolor="black", markersize=11, label="Maschine OK"),
            Line2D([0], [0], marker="s", color="w", markerfacecolor=ZUSTAND_FARBEN["WARM"],
                   markeredgecolor="black", markersize=11, label="WARM"),
            Line2D([0], [0], marker="s", color="w", markerfacecolor=ZUSTAND_FARBEN["HOT"],
                   markeredgecolor="black", markersize=11, label="HOT"),
            Line2D([0], [0], marker="s", color="w", markerfacecolor=EARLYWARNING_FARBE,
                   markeredgecolor="black", markersize=11, label="Fruehwarnung"),
            Line2D([0], [0], marker="P", color="w", markerfacecolor=WARTUNG_FARBE,
                   markeredgecolor="black", markersize=12, label="Wartung (W)"),
            Line2D([0], [0], marker="D", color="w", markerfacecolor=AUFTRAG_FARBE,
                   markeredgecolor="white", markersize=9, label="Auftrag"),
        ]
        ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.01, 1.0),
                  fontsize=8, framealpha=0.95)
        ax.grid(True, alpha=0.3)
    return post_process


model_ext = FactoryModelExtended(
    width=10, height=10, density=DENSITY, threshold=THRESHOLD,
    n_maintenance=N_MAINTENANCE, n_orders=N_ORDERS,
)

space_ext = make_space_component(
    agent_portrayal_ext,
    post_process=make_post_process_ext(model_ext),
)


@solara.component
def DiagrammPanel(model):
    """Alle Diagramme in EINER Figur (Subplots) -> kein Ueberlappen mehr."""
    update_counter.get()  # Re-Render pro Simulationsschritt
    df = model.datacollector.get_model_vars_dataframe()
    fig = Figure(figsize=(5.5, 8.5))
    ax1, ax2, ax3 = fig.subplots(3, 1)
    if len(df):
        # 1) Maschinenzustand (links) + Praeventionsrate auf rechter Achse
        ax1.plot(df.index, df["HotMachines"], color="#d62728", label="HOT")
        ax1.plot(df.index, df["WarmMachines"], color="#ff7f0e", label="WARM")
        ax1.plot(df.index, df["EarlyWarnings"], color="gold", label="Fruehwarnung")
        ax1.set_ylabel("Anzahl Maschinen")
        ax1.set_title("Zustand & Praeventionsrate", fontsize=9)
        axr = ax1.twinx()
        axr.plot(df.index, df["PraevRate"], color="green", linestyle="--", label="Praev.-Rate %")
        axr.set_ylabel("Praeventionsrate (%)")
        axr.set_ylim(0, 105)
        l1, la1 = ax1.get_legend_handles_labels()
        l2, la2 = axr.get_legend_handles_labels()
        ax1.legend(l1 + l2, la1 + la2, fontsize=6, loc="upper left")
        # 2) Temperaturtrend (Predictor-Sicht) mit Schwellwertlinie
        ax2.plot(df.index, df["AvgTemp"], color="#1f77b4", label="Ø Temp")
        ax2.plot(df.index, df["MaxTemp"], color="#d62728", label="Max Temp")
        ax2.axhline(model.threshold, color="gray", linestyle=":", label="HOT-Schwelle")
        ax2.set_ylabel("Temperatur (°C)")
        ax2.set_title("Temperaturtrend", fontsize=9)
        ax2.legend(fontsize=6, loc="upper left")
        # 3) Praeventions-Bilanz (kumuliert)
        ax3.plot(df.index, df["HotVerhindert"], color="green", label="verhindert")
        ax3.plot(df.index, df["HotPassiert"], color="#d62728", label="eingetreten")
        ax3.set_ylabel("kumuliert")
        ax3.set_xlabel("Schritt")
        ax3.set_title("HOT: verhindert vs. eingetreten", fontsize=9)
        ax3.legend(fontsize=6, loc="upper left")
    fig.tight_layout()
    solara.FigureMatplotlib(fig)


@solara.component
def ParamControl(label, reactive, apply, default, step, vmin, vmax, integer=False):
    """Eine Parameter-Zeile: Eingabefeld (min/max-begrenzt) + - / + / Reset-Knopf.

    Der Wert steckt in einer reaktiven Variable (solara.use_reactive in der
    Elternkomponente), damit ihn das per-Tick-Neuzeichnen NICHT zuruecksetzt.
    """
    def commit(v):
        try:
            v = float(v)
        except (TypeError, ValueError):
            return
        v = max(vmin, min(vmax, v))           # auf [min, max] begrenzen
        v = int(round(v)) if integer else round(v, 2)
        reactive.set(v)
        apply(v)

    with solara.Row(style={"align-items": "center", "gap": "4px"}):
        if integer:
            solara.InputInt(f"{label} [{vmin}-{vmax}]", value=reactive.value, on_value=commit)
        else:
            solara.InputFloat(f"{label} [{vmin}-{vmax}]", value=reactive.value, on_value=commit)
        solara.Button("−", on_click=lambda: commit(reactive.value - step))
        solara.Button("+", on_click=lambda: commit(reactive.value + step))
        solara.Button("↺", on_click=lambda: commit(default))


@solara.component
def LiveSteuerung(model):
    """Live-Steuerung des LAUFENDEN Modells: Eingabefelder + -/+ je Variable,
    Reset je Variable und global, Prädiktions-Schalter, zweispaltig."""
    d = model._param_defaults
    # Reaktive Zustaende (einmalig initialisiert -> ueberleben das Neuzeichnen)
    r_pred  = solara.use_reactive(model.use_prediction)
    r_maint = solara.use_reactive(count_maintenance(model))
    r_thr   = solara.use_reactive(float(model.threshold))
    r_cool  = solara.use_reactive(float(model.cooling_power))
    r_heat  = solara.use_reactive(float(model.order_heat))
    r_spawn = solara.use_reactive(float(model.order_spawn_rate))
    r_back  = solara.use_reactive(int(model.target_backlog))

    def set_pred(v):
        r_pred.set(v)
        model.use_prediction = v

    def reset_all():
        model.reset_parameters()
        r_pred.set(model.use_prediction)
        r_maint.set(count_maintenance(model))
        r_thr.set(float(model.threshold))
        r_cool.set(float(model.cooling_power))
        r_heat.set(float(model.order_heat))
        r_spawn.set(float(model.order_spawn_rate))
        r_back.set(int(model.target_backlog))

    with solara.Card("Live-Steuerung (wirkt sofort)"):
        solara.Switch(label="Praediktion aktiv (sonst rein reaktiv)",
                      value=r_pred.value, on_value=set_pred)
        with solara.Columns([1, 1]):
            with solara.Column():
                ParamControl("Wartungsagenten", r_maint,
                             lambda v: model.set_maintenance_count(int(v)),
                             default=model._default_n_maintenance, step=1, vmin=0, vmax=10, integer=True)
                ParamControl("HOT-Schwellwert °C", r_thr,
                             lambda v: setattr(model, "threshold", v),
                             default=d["threshold"], step=5, vmin=40, vmax=120)
                ParamControl("Kuehlleistung °C", r_cool,
                             lambda v: setattr(model, "cooling_power", v),
                             default=d["cooling_power"], step=1, vmin=5, vmax=40)
            with solara.Column():
                ParamControl("Auftrags-Hitze/Schritt", r_heat,
                             lambda v: setattr(model, "order_heat", v),
                             default=d["order_heat"], step=0.5, vmin=0.5, vmax=6.0)
                ParamControl("Auftrags-Spawnrate", r_spawn,
                             lambda v: setattr(model, "order_spawn_rate", v),
                             default=d["order_spawn_rate"], step=0.05, vmin=0.0, vmax=1.0)
                ParamControl("Ziel-Backlog", r_back,
                             lambda v: setattr(model, "target_backlog", int(v)),
                             default=d["target_backlog"], step=1, vmin=0, vmax=40, integer=True)
        solara.Button("Alle Werte zuruecksetzen", icon_name="mdi-restore", on_click=reset_all)


<IPython.core.display.Javascript object>

## 5. OPC UA HOT-Monitor (Lesen vom Server)

Der HOT-Monitor liest Temperaturen vom OPC UA-Server und setzt
`Mxx_RepairNeeded = True`, sobald eine Maschine HOT wird.

Er läuft als asyncio-Task parallel zur Simulation – beide teilen sich
den Jupyter Event-Loop, blockieren sich aber nicht gegenseitig.


In [7]:
hot_monitor_task = None

def start_hot_monitor_in_background(
    mode: str = "subscription",
    runtime_seconds: float = 300.0,
    interval: float = 1.0,
    publishing_interval_ms: int = 500,
):
    """OPC UA HOT-Monitor als asyncio-Hintergrundtask starten."""
    global hot_monitor_task
    loop = asyncio.get_event_loop()

    if mode == "polling":
        coro = hot_client.polling_hot_monitor(
            runtime_seconds=runtime_seconds,
            poll_interval=interval,
        )
    else:
        coro = hot_client.subscription_hot_monitor(
            runtime_seconds=runtime_seconds,
            publishing_interval_ms=publishing_interval_ms,
        )

    hot_monitor_task = loop.create_task(coro)
    print(f"HOT-Monitor gestartet ({mode}-Modus). Task:", hot_monitor_task)


def cancel_hot_monitor():
    """Laufenden HOT-Monitor-Task abbrechen."""
    global hot_monitor_task
    if hot_monitor_task is not None and not hot_monitor_task.done():
        hot_monitor_task.cancel()
        print("HOT-Monitor-Task abgebrochen.")
    else:
        print("Kein laufender HOT-Monitor-Task gefunden.")


## 6. OPC UA Writer-Task (Schreiben auf den Server)

Dieser Task ist das Herzstück der neuen Kopplung. Er:

1. **verbindet sich einmalig** mit dem OPC UA-Server,
2. **löst alle Maschinen-Nodes einmalig auf** und speichert sie im `node_cache`
   (das ist deutlich effizienter als bei jedem Schreibvorgang `get_child()` aufzurufen),
3. **liest periodisch** den `opcua_write_buffer` und schreibt die Werte auf den Server.

### Warum Node-Caching?

`get_child()` ist ein Netzwerkaufruf – bei jedem Schreibvorgang würde das
die Latenz unnötig erhöhen. Durch einmaliges Auflösen der Nodes beim Start
reduzieren wir jeden Schreibzyklus auf reine `write_value()`-Aufrufe.

### Adressraum-Mapping

Der OPC UA-Server legt Maschinen unter folgendem Pfad an:

```
Objects / Factory / Machines / M01 / Temperature
                                   / State
                                   / Busy
```

Der Namespace-Index `nsidx` wird dynamisch vom Server abgefragt –
keine hartcodierten Indizes.


In [8]:
from asyncua import Client

OPC_UA_URL    = "opc.tcp://localhost:4840/freeopcua/server/"
OPC_UA_NS_URI = "http://ostfalia.de/ipt/factory"

opcua_writer_task = None


async def opcua_writer(runtime_seconds: float = 300.0, interval: float = 1.0):
    """Überträgt opcua_write_buffer und beide Flag-Puffer auf den OPC UA-Server."""
    async with Client(url=OPC_UA_URL) as client:
        nsidx = await client.get_namespace_index(OPC_UA_NS_URI)
        print(f"OPC UA Writer verbunden. Namespace-Index: {nsidx}")

        # ── Maschinen-Nodes cachen ─────────────────────────────────────────
        node_cache: dict = {}
        for i in range(1, 51):
            name = f"M{i:02d}"
            base = f"0:Objects/{nsidx}:Factory/{nsidx}:Machines/{nsidx}:{name}"
            try:
                node_cache[name] = {
                    "Temperature": await client.nodes.root.get_child(f"{base}/{nsidx}:Temperature"),
                    "State":       await client.nodes.root.get_child(f"{base}/{nsidx}:State"),
                    "Busy":        await client.nodes.root.get_child(f"{base}/{nsidx}:Busy"),
                }
            except Exception:
                break
        print(f"OPC UA Writer: {len(node_cache)} Maschinen-Nodes gecacht.")

        jobs_base = f"0:Objects/{nsidx}:Factory/{nsidx}:Maintenance/{nsidx}:Jobs"

        # ── RepairNeeded-Nodes cachen ──────────────────────────────────────
        repair_cache: dict = {}
        for name in node_cache:
            try:
                repair_cache[name] = await client.nodes.root.get_child(
                    f"{jobs_base}/{nsidx}:{name}_RepairNeeded"
                )
            except Exception as e:
                print(f"RepairNeeded-Node für {name} nicht gefunden: {e}")

        # ── EarlyWarning-Nodes cachen ──────────────────────────────────────
        ew_cache: dict = {}
        for name in node_cache:
            try:
                ew_cache[name] = await client.nodes.root.get_child(
                    f"{jobs_base}/{nsidx}:{name}_EarlyWarning"
                )
            except Exception as e:
                print(f"EarlyWarning-Node für {name} nicht gefunden: {e}")

        print(f"OPC UA Writer: {len(repair_cache)} RepairNeeded / {len(ew_cache)} EarlyWarning Nodes gecacht.")

        # Alle Flags beim Start zurücksetzen
        for node in repair_cache.values():
            try: await node.write_value(False)
            except Exception: pass
        for node in ew_cache.values():
            try: await node.write_value(False)
            except Exception: pass

        # ── Schreibschleife ────────────────────────────────────────────────
        deadline = asyncio.get_event_loop().time() + runtime_seconds
        while asyncio.get_event_loop().time() < deadline:

            # Maschinenzustände schreiben
            for machine_name, values in list(opcua_write_buffer.items()):
                if machine_name not in node_cache:
                    continue
                nodes = node_cache[machine_name]
                try:
                    await nodes["Temperature"].write_value(float(values["Temperature"]))
                    await nodes["State"].write_value(str(values["State"]))
                    await nodes["Busy"].write_value(bool(values["Busy"]))
                except Exception as e:
                    print(f"Schreibfehler {machine_name}: {e}")

            # RepairNeeded-Flags zurücksetzen
            for machine_name, value in list(opcua_repair_buffer.items()):
                if machine_name not in repair_cache:
                    continue
                try:
                    await repair_cache[machine_name].write_value(value)
                    del opcua_repair_buffer[machine_name]
                except Exception as e:
                    print(f"RepairNeeded-Schreibfehler {machine_name}: {e}")

            # EarlyWarning-Flags zurücksetzen
            for machine_name, value in list(opcua_earlywarning_buffer.items()):
                if machine_name not in ew_cache:
                    continue
                try:
                    await ew_cache[machine_name].write_value(value)
                    del opcua_earlywarning_buffer[machine_name]
                except Exception as e:
                    print(f"EarlyWarning-Schreibfehler {machine_name}: {e}")

            await asyncio.sleep(interval)

        print("OPC UA Writer beendet.")


def start_opcua_writer(runtime_seconds: float = 300.0, interval: float = 1.0):
    global opcua_writer_task
    loop = asyncio.get_event_loop()
    opcua_writer_task = loop.create_task(
        opcua_writer(runtime_seconds=runtime_seconds, interval=interval)
    )
    print(f"OPC UA Writer gestartet (Intervall: {interval}s). Task:", opcua_writer_task)


def cancel_opcua_writer():
    global opcua_writer_task
    if opcua_writer_task is not None and not opcua_writer_task.done():
        opcua_writer_task.cancel()
        print("OPC UA Writer-Task abgebrochen.")
    else:
        print("Kein laufender Writer-Task gefunden.")

In [9]:
opcua_reader_task = None


async def opcua_reader(runtime_seconds: float = 300.0, interval: float = 1.0):
    """Liest EarlyWarning- und RepairNeeded-Flags vom Server; füllt server_flags.

    Läuft als asyncio-Hintergrundtask. MaintenanceAgent.step() liest server_flags
    synchron – kein Lock nötig, da asyncio kooperativ ist.
    """
    async with Client(url=OPC_UA_URL) as client:
        nsidx = await client.get_namespace_index(OPC_UA_NS_URI)
        print(f"OPC UA Reader verbunden. Namespace-Index: {nsidx}")

        jobs_base = f"0:Objects/{nsidx}:Factory/{nsidx}:Maintenance/{nsidx}:Jobs"
        ew_cache: dict = {}
        rn_cache: dict = {}

        for i in range(1, 51):
            name = f"M{i:02d}"
            try:
                ew_cache[name] = await client.nodes.root.get_child(
                    f"{jobs_base}/{nsidx}:{name}_EarlyWarning"
                )
                rn_cache[name] = await client.nodes.root.get_child(
                    f"{jobs_base}/{nsidx}:{name}_RepairNeeded"
                )
                server_flags[name] = {"EarlyWarning": False, "RepairNeeded": False}
            except Exception:
                break

        print(f"OPC UA Reader: {len(ew_cache)} Maschinen-Flags gecacht.")

        deadline = asyncio.get_event_loop().time() + runtime_seconds
        while asyncio.get_event_loop().time() < deadline:
            for name in list(ew_cache.keys()):
                try:
                    server_flags[name]["EarlyWarning"] = bool(await ew_cache[name].read_value())
                    server_flags[name]["RepairNeeded"] = bool(await rn_cache[name].read_value())
                except Exception as e:
                    print(f"Lesefehler {name}: {e}")
            await asyncio.sleep(interval)

        print("OPC UA Reader beendet.")


def start_opcua_reader(runtime_seconds: float = 300.0, interval: float = 1.0):
    """OPC UA Reader als asyncio-Hintergrundtask starten."""
    global opcua_reader_task
    loop = asyncio.get_event_loop()
    opcua_reader_task = loop.create_task(
        opcua_reader(runtime_seconds=runtime_seconds, interval=interval)
    )
    print(f"OPC UA Reader gestartet (Intervall: {interval}s). Task:", opcua_reader_task)


def cancel_opcua_reader():
    """Laufenden Reader-Task abbrechen."""
    global opcua_reader_task
    if opcua_reader_task is not None and not opcua_reader_task.done():
        opcua_reader_task.cancel()
        print("OPC UA Reader-Task abgebrochen.")
    else:
        print("Kein laufender Reader-Task gefunden.")

## 7. Simulation, HOT-Monitor und Writer gemeinsam starten

Jetzt laufen drei Komponenten parallel im selben Jupyter-Prozess:

| Komponente | Art | Aufgabe |
|---|---|---|
| `SolaraViz` | synchron (Hauptthread) | Mesa-Simulation visualisieren |
| `subscription_hot_monitor` | asyncio-Task | Temperaturen vom Server lesen, RepairNeeded setzen |
| `opcua_writer` | asyncio-Task | Puffer auf den Server schreiben |

**Reihenfolge:** Zuerst Writer und Monitor starten, dann `page_ext` als
letzte Zellenausgabe – sonst blockiert SolaraViz den Event-Loop bevor
die Tasks registriert sind.


In [10]:
# OPC UA Writer starten (schreibt Mesa-Zustaende + Flag-Puffer auf den Server)
start_opcua_writer(runtime_seconds=300, interval=1.0)

# OPC UA Reader starten (liest EarlyWarning + RepairNeeded -> server_flags)
start_opcua_reader(runtime_seconds=300, interval=1.0)

# HOT-Monitor starten (liest vom Server, setzt RepairNeeded-Flags)
start_hot_monitor_in_background(mode="subscription", runtime_seconds=300)


# Steuerpanel + Diagramme in EINER Komponente (rechte Spalte). Dadurch hat die
# Seite nur ZWEI Rasterkacheln (Raster links, rechte Spalte rechts) -> es gibt
# keine dritte Kachel darunter, die ueberlagert werden koennte. Alles auf einem Tab.
@solara.component
def RechteSpalte(model):
    LiveSteuerung(model)
    DiagrammPanel(model)


# Mesa-Simulation starten: links das Raster, rechts Steuerung + Diagramme.
page_ext = SolaraViz(
    model_ext,
    components=[space_ext, RechteSpalte],
    name="Erweitertes Fabrikmodell mit OPC UA",
    play_interval=200,
)


### OPC UA Ausgaben anzeigen


In [11]:
display(opcua_out)

Output(layout=Layout(border_bottom='1px solid gray', border_left='1px solid gray', border_right='1px solid gra…

In [12]:
page_ext

Cannot show ipywidgets in text

## 8. Tasks beenden

Mit den folgenden Zellen können die Hintergrundtasks jederzeit manuell
abgebrochen werden – z.B. um Parameter zu ändern und neu zu starten.


In [13]:
# cancel_hot_monitor()
# cancel_opcua_writer()
# cancel_opcua_reader()

## 9. Zusammenfassung und Erweiterungsmöglichkeiten

### Was wurde umgesetzt?

- **MachineAgent** schreibt nach jedem `step()` Temperatur, Zustand und Busy-Flag
  in den `opcua_write_buffer`.
- **`opcua_writer()`** überträgt den Puffer periodisch auf den OPC UA-Server –
  vollständig entkoppelt vom Mesa-Scheduler.
- **HOT-Monitor** liest Temperaturen vom Server und setzt `RepairNeeded`-Flags.

### Architekturprinzip: Lese/Schreibe-Entkopplung

```
Mesa (synchron)          Puffer          OPC UA (async)
─────────────────        ──────          ──────────────
step() → schreiben  ──►  dict    ──►  write_value()
                                  ◄──  read_value() (HOT-Monitor)
```


